# 0420 · Interpretabilidad y hallazgos operativos (OE5)

**Qué hace.** Interpreta los modelos finales que fijó `0410_EvaluacionTemporal_2`, sin volver a ajustar nada:
1. **Estabilidad** de la importancia por permutación entre los bloques de validación (¿las mismas variables pesan en abril, mayo y junio?).
2. **Importancia por permutación** del modelo final sobre la prueba.
3. **Valores SHAP** agregados por variable original, con la dirección del efecto y gráficos de dependencia.
4. **Traducción operativa**: carga de alertas por día, por categoría de zona y franja, aciertos por segmento, motivos principales de las alertas y márgenes de error del tiempo estimado.

**Lectura.** Las importancias y los SHAP describen **cómo usa el modelo** las variables; no son efectos causales (realimentación OE5: lectura no causal). La interpretación de la realimentación se hace en validación durante el desarrollo y, sobre la prueba, **solo del modelo final ya fijado**.

**Requisitos.** Haber ejecutado la prueba final de `0410_EvaluacionTemporal_2` (carpeta `MODELOS/` y `P4b_decision.json`).

**Salidas.** CSV y PNG en `REVISION_2026_10/INTERPRETACION/`. Solo agregados; ninguna guía individual.

In [ ]:
# Si faltan librerías:
# %pip install shap lightgbm xgboost scikit-learn joblib

In [ ]:
# CONFIGURACIÓN
import os, json, numpy as np, pandas as pd, joblib, shap, matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.inspection import permutation_importance
PATH_DATABRICKS = "/Volumes/proyectogrado/source/sourcedb/"
CARPETA_SALIDAS = f"{PATH_DATABRICKS}LOG_CONSOLIDADO/REVISION_2026_10/"
CARPETA_MODELOS = f"{CARPETA_SALIDAS}MODELOS/"
CARPETA_INT     = f"{CARPETA_SALIDAS}INTERPRETACION/"
os.makedirs(CARPETA_INT, exist_ok=True)
SEMILLA = 42
N_SHAP = 2000            # filas de prueba para SHAP (Random Forest es lento: bajar a 1000 si tarda)
N_PERMUTACION = 20_000   # filas de prueba para la importancia por permutación
TOP = 15
DEC = json.load(open(f"{CARPETA_SALIDAS}P4b_decision.json"))
CAT, NUM = DEC["variables_categoricas"], DEC["variables_numericas"]
print(json.dumps({k: DEC[k] for k in ["modelo_clf", "modelo_reg", "calibracion", "umbral", "alertas_por_dia_escenario"]}, indent=1))

## Funciones (no es necesario modificarlas)

In [ ]:
def variable_original(nombre_transformado):
    """'cat__categoria_zona_C' -> 'categoria_zona'; 'num__sla_h' -> 'sla_h'."""
    t, resto = nombre_transformado.split("__", 1)
    if t == "num": return resto
    return max([c for c in CAT if resto.startswith(c + "_")], key=len)

def shap_modelo(pipe, X, X_fondo, tarea):
    """SHAP sobre las columnas transformadas; devuelve (matriz, nombres, datos transformados)."""
    prep, mod = pipe.named_steps["prep"], pipe.named_steps["modelo"]
    densa = lambda Z: Z.toarray() if hasattr(Z, "toarray") else np.asarray(Z)
    Xt = densa(prep.transform(X)).astype(float); nombres = list(prep.get_feature_names_out())
    if mod.__class__.__name__ in ("LogisticRegression", "LinearRegression"):
        exp = shap.LinearExplainer(mod, densa(prep.transform(X_fondo)).astype(float))
        sv = exp.shap_values(Xt)
    else:
        sv = shap.TreeExplainer(mod).shap_values(Xt, check_additivity=False)
    if isinstance(sv, list): sv = sv[1] if len(sv) == 2 else sv[0]
    sv = np.asarray(sv)
    if sv.ndim == 3: sv = sv[:, :, 1]
    return sv, nombres, Xt

def agregar_por_variable(sv, nombres):
    orig = [variable_original(n) for n in nombres]
    df = pd.DataFrame(sv, columns=nombres)
    return df.T.groupby(orig).sum().T          # suma de las columnas de una misma variable (one-hot)

## 1. Estabilidad de la importancia entre bloques de validación

In [ ]:
imp = pd.read_csv(f"{CARPETA_SALIDAS}P4b_importancia_pliegues.csv")
filas = []
for (tarea, modelo), g in imp.groupby(["tarea", "modelo"]):
    piv = g.pivot_table(index="variable", columns="bloque", values="importancia")
    bloques = list(piv.columns)
    rhos = [spearmanr(piv[a], piv[b]).correlation for i, a in enumerate(bloques) for b in bloques[i + 1:]]
    top = piv.mean(axis=1).sort_values(ascending=False)
    en_top5 = (piv.rank(ascending=False) <= 5).sum(axis=1)
    filas.append({"tarea": tarea, "modelo": modelo, "spearman_min_entre_bloques": np.nanmin(rhos),
                  "spearman_medio": np.nanmean(rhos), "top5_promedio": ", ".join(top.index[:5]),
                  "variables_top5_en_todos_los_bloques": ", ".join(en_top5[en_top5 == len(bloques)].index)})
    piv.assign(media=piv.mean(axis=1)).sort_values("media", ascending=False).round(5) \
       .to_csv(f"{CARPETA_INT}01_permutacion_validacion_{tarea}_{modelo}.csv")
estabilidad = pd.DataFrame(filas); display(estabilidad)
estabilidad.to_csv(f"{CARPETA_INT}01_estabilidad_importancias.csv", index=False)
print("Spearman cercano a 1: el ordenamiento de variables se mantiene entre bloques; por debajo de 0,5: interpretar con cautela.")

## 2. Importancia por permutación del modelo final (prueba)

In [ ]:
modelos, datos = {}, {}
for tarea in ["clf", "reg"]:
    modelos[tarea] = joblib.load(f"{CARPETA_MODELOS}modelo_final_{tarea}.joblib")
    datos[tarea] = pd.read_parquet(f"{CARPETA_MODELOS}prueba_{tarea}.parquet")
perm = []
for tarea, scoring in [("clf", "average_precision"), ("reg", "neg_mean_absolute_error")]:
    d = datos[tarea].sample(min(N_PERMUTACION, len(datos[tarea])), random_state=SEMILLA)
    r = permutation_importance(modelos[tarea], d[CAT + NUM], d["y"], n_repeats=5, random_state=SEMILLA, scoring=scoring, n_jobs=1)
    t = pd.DataFrame({"tarea": tarea, "variable": CAT + NUM, "importancia": r.importances_mean, "desv": r.importances_std}) \
          .sort_values("importancia", ascending=False)
    perm.append(t)
    fig, ax = plt.subplots(figsize=(7, 5)); tt = t.head(TOP)[::-1]
    ax.barh(tt.variable, tt.importancia, xerr=tt.desv, color="#4C78A8")
    ax.set_title(f"Permutación en prueba · {DEC['modelo_' + tarea]} · {'pérdida de AP' if tarea == 'clf' else 'aumento del MAE (h)'}")
    plt.tight_layout(); fig.savefig(f"{CARPETA_INT}02_permutacion_prueba_{tarea}.png", dpi=130); plt.show()
perm = pd.concat(perm); perm.to_csv(f"{CARPETA_INT}02_permutacion_prueba.csv", index=False)
display(perm.groupby("tarea").head(10).round(4))

## 3. Valores SHAP

SHAP reparte cada predicción entre las variables. Para una variable categórica se suman sus columnas *one-hot*. En clasificación, los valores están en la escala del modelo (log-odds en los ensambles de *boosting* y en la logística; probabilidad en Random Forest). La **dirección** de una variable numérica es la correlación de Spearman entre su valor y su SHAP: positiva significa «a mayor valor, mayor riesgo o más horas».

In [ ]:
resumen_shap, SHAP = [], {}
for tarea in ["clf", "reg"]:
    d = datos[tarea].sample(min(N_SHAP, len(datos[tarea])), random_state=SEMILLA)
    fondo = pd.read_parquet(f"{CARPETA_MODELOS}muestra_entrenamiento_{tarea}.parquet").sample(500, random_state=SEMILLA)
    sv, nombres, Xt = shap_modelo(modelos[tarea], d[CAT + NUM], fondo[CAT + NUM], tarea)
    agg = agregar_por_variable(sv, nombres); agg.index = d.index
    SHAP[tarea] = (d, agg)
    filas = []
    for v in agg.columns:
        direccion = spearmanr(d[v], agg[v], nan_policy="omit").correlation if v in NUM else np.nan
        filas.append({"tarea": tarea, "variable": v, "media_abs_shap": agg[v].abs().mean(), "direccion_spearman": direccion})
    t = pd.DataFrame(filas).sort_values("media_abs_shap", ascending=False); resumen_shap.append(t)
    fig, ax = plt.subplots(figsize=(7, 5)); tt = t.head(TOP)[::-1]
    ax.barh(tt.variable, tt.media_abs_shap, color=np.where(tt.direccion_spearman > 0, "#E45756", np.where(tt.direccion_spearman < 0, "#4C78A8", "#999")))
    ax.set_title(f"SHAP medio absoluto · {DEC['modelo_' + tarea]} ({tarea})\nrojo: aumenta con la variable · azul: disminuye · gris: categórica")
    plt.tight_layout(); fig.savefig(f"{CARPETA_INT}03_shap_barras_{tarea}.png", dpi=130); plt.show()
    plt.figure(); shap.summary_plot(sv, Xt, feature_names=nombres, max_display=TOP, show=False)
    plt.title(f"SHAP por columna transformada ({tarea})"); plt.tight_layout()
    plt.savefig(f"{CARPETA_INT}03_shap_resumen_{tarea}.png", dpi=130, bbox_inches="tight"); plt.show()
resumen_shap = pd.concat(resumen_shap); resumen_shap.to_csv(f"{CARPETA_INT}03_shap_por_variable.csv", index=False)
display(resumen_shap.groupby("tarea").head(10).round(4))

In [ ]:
# Dependencia: las 3 variables numéricas con mayor SHAP medio en cada tarea
for tarea in ["clf", "reg"]:
    d, agg = SHAP[tarea]
    top_num = [v for v in resumen_shap[resumen_shap.tarea == tarea].variable if v in NUM][:3]
    fig, axs = plt.subplots(1, len(top_num), figsize=(5 * len(top_num), 3.6), squeeze=False)
    for ax, v in zip(axs[0], top_num):
        ax.scatter(d[v], agg[v], s=4, alpha=0.4, color="#4C78A8"); ax.axhline(0, color="k", lw=0.8)
        ax.set_xlabel(v); ax.set_ylabel("SHAP")
    fig.suptitle(f"Dependencia SHAP · {tarea}"); plt.tight_layout()
    fig.savefig(f"{CARPETA_INT}04_dependencia_{tarea}.png", dpi=130); plt.show()
# Efecto medio de las categóricas (SHAP medio por categoría)
filas = []
for tarea in ["clf", "reg"]:
    d, agg = SHAP[tarea]
    for v in CAT:
        g = agg[v].groupby(d[v].astype(str))
        for cat, s in g:
            if len(s) >= 30: filas.append({"tarea": tarea, "variable": v, "categoria": cat, "n": len(s), "shap_medio": s.mean()})
efectos_cat = pd.DataFrame(filas).sort_values(["tarea", "variable", "shap_medio"])
efectos_cat.to_csv(f"{CARPETA_INT}04_shap_por_categoria.csv", index=False); display(efectos_cat.round(4).head(30))

## 4. Traducción operativa

In [ ]:
# 4.1 Carga de alertas con el umbral fijado en validación
c = datos["clf"].copy(); c["alerta"] = c.pred >= DEC["umbral"]
dias = c.dia.nunique()
carga = pd.DataFrame({"alertas_por_dia_media": [c.groupby("dia").alerta.sum().mean()],
                      "alertas_por_dia_p90": [c.groupby("dia").alerta.sum().quantile(0.9)],
                      "precision": [c.loc[c.alerta, "y"].mean()],
                      "exhaustividad": [c.loc[c.y == 1, "alerta"].mean()],
                      "incumplimientos_no_alertados_por_dia": [((c.y == 1) & ~c.alerta).sum() / dias],
                      "escenario_capacidad": [DEC["alertas_por_dia_escenario"]]})
display(carga.round(3)); carga.to_csv(f"{CARPETA_INT}05_carga_alertas.csv", index=False)
seg = []
for col in ["categoria_zona", "franja", "dia_semana", "regional_origen"]:
    g = c.groupby(col)
    t = pd.DataFrame({"guias": g.size(), "alertas_por_dia": g.alerta.sum() / dias, "tasa_real": g.y.mean(),
                      "precision_alertas": g.apply(lambda x: x.loc[x.alerta, "y"].mean() if x.alerta.any() else np.nan),
                      "exhaustividad": g.apply(lambda x: x.loc[x.y == 1, "alerta"].mean() if (x.y == 1).any() else np.nan)})
    t = t[t.guias >= 30]; t.insert(0, "segmento", col); seg.append(t.reset_index().rename(columns={col: "valor"}))
seg = pd.concat(seg); display(seg.round(3)); seg.to_csv(f"{CARPETA_INT}05_alertas_por_segmento.csv", index=False)

In [ ]:
# 4.2 Motivos de las alertas: variable con mayor contribución positiva al riesgo en cada guía alertada (muestra SHAP)
d, agg = SHAP["clf"]
alertadas = d.pred >= DEC["umbral"]
if alertadas.any():
    motivo = agg[alertadas].idxmax(axis=1)
    motivos = motivo.value_counts(normalize=True).rename("proporcion_de_alertas").to_frame()
    motivos["n"] = motivo.value_counts()
    display(motivos.round(3)); motivos.to_csv(f"{CARPETA_INT}06_motivos_alertas.csv")
else:
    print("Ninguna guía de la muestra SHAP supera el umbral.")
# 4.3 Margen de error del tiempo estimado, por categoría de zona y franja (para comunicar la estimación con su rango)
r = datos["reg"].copy(); r["error_abs"] = (r.y - r.pred).abs(); r["subestima"] = r.y > r.pred
banda = r.groupby(["categoria_zona", "franja"]).agg(guias=("y", "size"), horas_reales_mediana=("y", "median"),
          MAE_h=("error_abs", "mean"), P80_error_h=("error_abs", lambda s: s.quantile(0.8)),
          P90_error_h=("error_abs", lambda s: s.quantile(0.9)), pct_subestima=("subestima", "mean"))
banda = banda[banda.guias >= 30]
display(banda.round(2)); banda.to_csv(f"{CARPETA_INT}07_banda_error_tiempo.csv")

## Cómo leer y qué entregar

- **Estabilidad (01):** si una variable está en el top 5 en todos los bloques, su papel es robusto; si el Spearman entre bloques es bajo, la interpretación debe ser prudente.
- **Permutación (02) y SHAP (03–04):** las variables que coinciden en los dos métodos y en la estabilidad son las que se reportan como «factores de mayor peso predictivo». Comparar con el ordenamiento descriptivo de `0210` (sección 7).
- **Operación (05–07):** cuántas alertas por día generaría el modelo con el umbral fijado, dónde se concentran, cuántos incumplimientos se escapan, por qué se alerta y con qué margen de error se comunica el tiempo estimado.
- **No causal:** «el modelo asigna más riesgo cuando el margen de SLA en T0 es menor» es correcto; «reducir X causaría menos incumplimientos» no se puede afirmar con este diseño.

**Entregar:** la carpeta `INTERPRETACION/` completa.